# Implementace

**FAO MULTIPLEX TRADE NETWORK**

Zde analyzujeme různé typy obchodních vztahů mezi státy, získané z FAO (Organizace pro výživu a zemědělství OSN). Celosvětová síť importu/exportu potravin je ekonomická síť, kde vrstvy představují produkty, uzly jsou státy a hrany v každé vrstvě reprezentují vztahy importu/exportu konkrétního potravinového produktu mezi státy. Data byla získána z FAO a byla vytvořena vícevrstvá síť odpovídající obchodování v roce 2010.  

Multiplexní síť použitá v článku je podmnožinou úplné datové sady, která obsahuje 364 vrstev.

**Reference:**  
M. De Domenico, V. Nicosia, A. Arenas, a V. Latora - "Structural reducibility of multilayer networks" - Nature Communications 2015 6, 6864  
Originální data: [https://manliodedomenico.com/data.php](https://manliodedomenico.com/data.php)

**Formát souborů:**  
layerID nodeID nodeID weight  
364 vrstev Multiplex

**Uzly:** 214  
**Hrany:** 318 346  
**Typ:** Finanční, multiplexní, orientovaná, ohodnocená

## Načtení knihoven

In [1]:
import random
from pathlib import Path
import os

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
from sklearn.cluster import AgglomerativeClustering

import warnings
from scipy.cluster.hierarchy import ClusterWarning

In [2]:
random.seed(42)
np.random.seed(42)

In [3]:
SAVE_DIR = Path("../results/implementation")
SAVE_DIR.mkdir(parents=True, exist_ok=True)

PLOT_DIR = SAVE_DIR / "plots"
PLOT_DIR.mkdir(parents=True, exist_ok=True)

## Načtení datasetu

In [4]:
layers_path = Path("../data/FAO_Multiplex_Trade/fao_trade_layers.txt")
nodes_path = Path("../data/FAO_Multiplex_Trade/fao_trade_nodes.txt")
edges_path = Path("../data/FAO_Multiplex_Trade/fao_trade_multiplex.edges")

layers_df = pd.read_csv(layers_path, sep='\\s+')
nodes_df = pd.read_csv(nodes_path, sep='\\s+')
edges_df = pd.read_csv(edges_path, sep='\\s+',
                    names=['layerID', 'source', 'target', 'weight'])

## Výpis několika náhodných vrstev

In [5]:
layers_df.sample(10, random_state=13)['layerLabel']

214    Lettuce_and_chicory
251          Oil,_rapeseed
27             Food_wastes
61          Cake,_rapeseed
313                    Rye
330                Vanilla
248     Offals,_liver_duck
101           Cocoa,_paste
73            Eggs,_liquid
345               Rapeseed
Name: layerLabel, dtype: object

In [6]:
summary = pd.DataFrame({
    'Počet uzlů': [nodes_df.shape[0]],
    'Počet vrstev': [layers_df.shape[0]],
    'Počet hran': [edges_df.shape[0]]
})
summary

,Počet uzlů,Počet vrstev,Počet hran
0,214,364,318346


# Redukce na menší počet vrstev pomocí shlukování

## Vytvoření uzlů a hran pro jednotlivé vrstvy

In [7]:
layer_edges_path = SAVE_DIR / "layer_edges.csv"

# AI made this part of saving results so I feel inferior to the inner machinations of AI overlords
if layer_edges_path.exists():
    print("Loading precomputed layer edges...")
    layer_edges = {}
    df = pd.read_csv(layer_edges_path)
    for _, row in df.iterrows():
        layer_id = row['layerID']
        # Convert string representation of edges back to set of tuples
        edges = set(tuple(map(float, e.split('-'))) for e in row['edges'].split(';') if e)
        layer_edges[layer_id] = edges
else:
    print("CSV not found, computing layer edges...")
    layer_edges = {
        layer_id: set(
            tuple(row[['source', 'target']])
            for _, row in edges_df[edges_df['layerID'] == layer_id].iterrows()
        )
        for layer_id in layers_df['layerID']
    }
    # Save to CSV
    records = []
    for layer_id, edges in layer_edges.items():
        edge_str = ';'.join(f"{int(s)}-{int(t)}" for s, t in edges)
        records.append({'layerID': layer_id, 'edges': edge_str})
    pd.DataFrame(records).to_csv(layer_edges_path, index=False)

CSV not found, computing layer edges...


## Vypočet Jaccardovy vzdálenosti mezi vrstvami

In [8]:
layer_ids = list(layer_edges.keys())
n_layers = len(layer_ids)
jaccard_dist = np.zeros((n_layers, n_layers))

for i in range(n_layers):
    for j in range(n_layers):
        if i == j:
            jaccard_dist[i, j] = 0
        else:
            a = layer_edges[layer_ids[i]]
            b = layer_edges[layer_ids[j]]
            intersection = len(a & b)
            union = len(a | b)
            jaccard_dist[i, j] = 1 - intersection / union if union > 0 else 1

## Shlukování vrstev pomocí `Agglomerative` Clustering

In [9]:
warnings.filterwarnings("ignore", category=ClusterWarning) # some useless warning about similarity
clustering = AgglomerativeClustering(n_clusters=5, linkage='complete')
labels = clustering.fit_predict(jaccard_dist)

In [10]:
layers_df['jaccard_cluster'] = labels

## Agregace hran podle Jaccardových shluků vrstev


In [11]:
edges_with_cluster = edges_df.merge(layers_df[['layerID', 'jaccard_cluster']], on='layerID')
aggregated_edges_jaccard = edges_with_cluster.groupby(
    ['source', 'target', 'jaccard_cluster']
).agg({'weight': 'sum'}).reset_index()

In [12]:
jaccard_summary = layers_df.groupby('jaccard_cluster').agg(
    layer_count=('layerID', 'count'),
    example_labels=('layerLabel', lambda x: ', '.join(x.sample(min(3, len(x)))))
).reset_index()
jaccard_summary

,jaccard_cluster,layer_count,example_labels
0,0,44,"Tea,_mate_extracts, Pet_food, Glucose_and_dext..."
1,1,73,"Fat,_liver_prepared_(foie_gras), Oils,_fats_of..."
2,2,54,"Skins,_sheep,_dry_salted, Meat,_beef_and_veal_..."
3,3,91,"Lentils, Nutmeg,_mace_and_cardamoms, Molasses"
4,4,102,"Meat,_beef,_preparations, Offals,_pigs,_edible..."


## Aplikace nových kategorií vrstev na původní data

In [13]:
category_counts = layers_df['jaccard_cluster'].value_counts()
category_counts

jaccard_cluster
4    102
3     91
1     73
2     54
0     44
Name: count, dtype: int64

## Kombinace hran a váh podle nových kategorií vrstev

In [14]:
edges_with_category = edges_df.merge(layers_df[['layerID', 'jaccard_cluster']], on='layerID')

aggregated_edges = edges_with_category.groupby(
    ['source', 'target', 'jaccard_cluster']
).agg({
    'weight': 'sum',  # Sum weights across all layers in category
}).reset_index()

In [15]:
print(f"Aggregated edges: {len(aggregated_edges):,}")
print(f"Number of categories: {aggregated_edges['jaccard_cluster'].nunique()}")

Aggregated edges: 36,870
Number of categories: 5


## Rozdíl počtu hran před a po agregaci

In [16]:
original_edge_count = len(edges_df)
aggregated_edge_count = len(aggregated_edges)
reduction = (original_edge_count - aggregated_edge_count) / original_edge_count * 100

print(f"Original edge count: {original_edge_count:,}")
print(f"Aggregated edge count: {aggregated_edge_count:,}")
print(f"Reduction of edges: {reduction:.2f}%")

Original edge count: 318,346
Aggregated edge count: 36,870
Reduction of edges: 88.42%


## Ukázka několika náhodných hran po agregaci podle kategorií vrstev

In [17]:
aggregated_edges.head()

,source,target,jaccard_cluster,weight
0,1,2,0,28.0
1,1,2,4,33.0
2,1,4,4,59.0
3,1,5,4,30.0
4,1,6,0,159.0


## Vytvoření `networkx` grafu z agregovaných hran

In [18]:
category_graphs = {}

# Prepare a mapping from nodeID to nodeLabel for fast lookup
node_id_to_label = nodes_df.set_index('nodeID')['nodeLabel'].to_dict()

for category in aggregated_edges['jaccard_cluster'].unique():
    cat_edges = aggregated_edges[aggregated_edges['jaccard_cluster'] == category]

    G = nx.DiGraph()
    
    # Přidat uzly podle všech zdrojů a cílů v této kategorii
    node_ids = set(cat_edges['source']).union(set(cat_edges['target']))
    for node_id in node_ids:
        label = node_id_to_label.get(node_id, str(node_id))
        G.add_node(node_id, label=label)
    
    # Přidat hrany
    for _, edge in cat_edges.iterrows():
        G.add_edge(edge['source'], edge['target'], weight=edge['weight'])
    
    category_graphs[category] = G

## Statistiky nově vytvořených grafů podle kategorií vrstev

In [19]:
stats = []
for category, graph in category_graphs.items():
    weights = [d['weight'] for _, _, d in graph.edges(data=True)]
    mean = np.mean(weights)
    median = np.median(weights)
    q1 = np.percentile(weights, 25)
    q3 = np.percentile(weights, 75)
    
    stats.append({
        'Category': category,
        'Mean': mean,
        'Median': median,
        'Q1': q1,
        'Q3': q3
    })

pd.DataFrame(stats)

,Category,Mean,Median,Q1,Q3
0,0,30426.147213,476.0,42.00,5101.00
1,4,44207.535175,834.0,63.25,8864.25
2,2,2647.696243,109.0,12.00,933.50
3,3,32717.292482,595.0,60.00,5402.00
4,1,13253.981530,284.5,30.00,2923.25


In [20]:
# print all jaccard clusters with their layer labels
for cluster_id in sorted(layers_df['jaccard_cluster'].unique()):
    cluster_layers = layers_df[layers_df['jaccard_cluster'] == cluster_id]['layerLabel'].tolist()
    print(f"Jaccard Cluster {cluster_id}:")
    for layer in cluster_layers[:10]:
        print(f"  - {layer}")
    print()

Jaccard Cluster 0:
  - Beverages,_non_alcoholic
  - Food_prep_nes
  - Cheese,_whole_cow_milk
  - Chocolate_products_nes
  - Flour,_wheat
  - Fat,_nes,_prepared
  - Beer_of_barley
  - Chillies_and_peppers,_dry
  - Crude_materials
  - Food_preparations,_flour,_malt_extract

Jaccard Cluster 1:
  - Cattle
  - Cake,_sunflower
  - Flour,_mixed_grain
  - Flax_fibre_and_tow
  - Alfalfa_meal_and_pellets
  - Barley,_pearled
  - Bran,_maize
  - Bran,_wheat
  - Cake,_rapeseed
  - Cranberries

Jaccard Cluster 2:
  - Feed,_compound,_nes
  - Cake,_palm_kernel
  - Cottonseed
  - Feed_supplements
  - Flax_fibre_raw
  - Cotton_linter
  - Animals_live_nes
  - Ducks
  - Beehives
  - Cake,_copra

Jaccard Cluster 3:
  - Cotton_lint
  - Bananas
  - Forage_products
  - Cake,_soybeans
  - Cloves
  - Cinnamon_(canella)
  - Coffee,_green
  - Cashew_nuts,_shelled
  - Apricots,_dry
  - Beeswax

Jaccard Cluster 4:
  - Cream_fresh
  - Cigarettes
  - Eggs,_hen,_in_shell
  - Chickens
  - Beans,_dry
  - Anise,_badian,_

In [21]:
cluster_names = {
    0: "Processed Foods & Beverages",
    1: "Livestock, Specialty Grains & Byproducts",
    2: "Animal Feed & Agricultural Byproducts",
    3: "Raw Commodity Crops & Primary Processing",
    4: "Fresh Foods & Dairy Products"
}

layers_df['cluster_name'] = layers_df['jaccard_cluster'].map(cluster_names)

In [22]:
layers_df.head()

,layerID,layerLabel,jaccard_cluster,cluster_name
0,1,"Beverages,_non_alcoholic",0,Processed Foods & Beverages
1,2,Cream_fresh,4,Fresh Foods & Dairy Products
2,3,Food_prep_nes,0,Processed Foods & Beverages
3,4,"Cheese,_whole_cow_milk",0,Processed Foods & Beverages
4,5,Cigarettes,4,Fresh Foods & Dairy Products


# Projekce

## Nevážená projekce

## Vážená projekce

# Metriky

# Náhodná procházka

# Komunity

# Vizualizace

## Základní statistky

## Komunity

## Náhodná procházka